In [18]:
# === torch_zamiennik.ipynb ===

import os
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
from sklearn.decomposition import PCA
from joblib import dump
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# === Foldery ===
os.makedirs('models', exist_ok=True)
os.makedirs('logs', exist_ok=True)
os.makedirs('reports', exist_ok=True)

# === Dane ===
data = pd.read_excel('złączone_dane.xlsx')
data = data.drop('image_id', axis=1)
data = data.drop(columns=[col for col in data.columns if any(x in col for x in ['3_p', '4_p', '5_p'])])

X = data.drop('label', axis=1)
y = data['label']
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# === PCA grupowe ===
def apply_grouped_pca(X, n_components=1):
    lm_0_cols = [col for col in X.columns if col.startswith('0_point_lm_')]
    lm_1_cols = [col for col in X.columns if col.startswith('1_point_lm_')]
    lm_2_cols = [col for col in X.columns if col.startswith('2_point_lm_')]
    vec_cols = [col for col in X.columns if '_vec_' in col]

    def pca_transform(cols, prefix):
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X[cols])
        pca = PCA(n_components=n_components)
        X_pca = pca.fit_transform(X_scaled)
        return pd.DataFrame(X_pca, columns=[f'{prefix}_pca_{i}' for i in range(n_components)], index=X.index)

    pca_lm_0 = pca_transform(lm_0_cols, '0')
    pca_lm_1 = pca_transform(lm_1_cols, '1')
    pca_lm_2 = pca_transform(lm_2_cols, '2')
    vec_features = X[vec_cols].reset_index(drop=True)

    return pd.concat([pca_lm_0, pca_lm_1, pca_lm_2, vec_features], axis=1)

X_pca = apply_grouped_pca(X)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_pca)

# === Model ===
class MLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(MLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.Tanh(),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.model(x)

# === CV ===
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
acc_list, prec_list, rec_list, f1_list = [], [], [], []

start_time = time.time()

X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y_encoded, dtype=torch.long)

for fold, (train_idx, val_idx) in enumerate(cv.split(X_scaled, y_encoded), 1):
    print(f"\n🔁 Fold {fold}")
    model = MLP(input_dim=X_scaled.shape[1], num_classes=len(np.unique(y_encoded)))
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    train_dataset = TensorDataset(X_tensor[train_idx], y_tensor[train_idx])
    val_dataset = TensorDataset(X_tensor[val_idx], y_tensor[val_idx])

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

    best_loss = float('inf')
    patience = 5
    wait = 0

    for epoch in range(50):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            output = model(xb)
            loss = criterion(output, yb)
            loss.backward()
            optimizer.step()

        # Walidacja
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                output = model(xb)
                loss = criterion(output, yb)
                val_loss += loss.item()

        val_loss /= len(val_loader)
        if val_loss < best_loss:
            best_loss = val_loss
            best_model = model.state_dict()
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_model)
    model.eval()
    y_pred = []
    with torch.no_grad():
        for xb in val_loader:
            preds = model(xb[0])
            y_pred.extend(torch.argmax(preds, dim=1).cpu().numpy())

    acc = accuracy_score(y_encoded[val_idx], y_pred)
    prec = precision_score(y_encoded[val_idx], y_pred, average='macro')
    rec = recall_score(y_encoded[val_idx], y_pred, average='macro')
    f1 = f1_score(y_encoded[val_idx], y_pred, average='macro')

    print(f"✅ Fold {fold} — Acc: {acc:.4f}, Prec: {prec:.4f}, Rec: {rec:.4f}, F1: {f1:.4f}")
    acc_list.append(acc)
    prec_list.append(prec)
    rec_list.append(rec)
    f1_list.append(f1)

# === Podsumowanie ===
training_time = time.time() - start_time
avg_acc = np.mean(acc_list)
avg_prec = np.mean(prec_list)
avg_rec = np.mean(rec_list)
avg_f1 = np.mean(f1_list)

# === Zapis ===
torch.save(model.state_dict(), 'models/torch_ssn_model.pt')
np.save('models/label_encoder_classes_pt.npy', le.classes_)
dump(scaler, 'models/scaler_pt.pkl')
report = classification_report(y_encoded, torch.argmax(model(X_tensor), dim=1).numpy(), digits=4)

with open('reports/torch_ssn_report.txt', 'w', encoding='utf-8') as f:
    f.write("Model: PyTorch MLP SSN\n")
    f.write("=== Raport klasyfikacji ===\n")
    f.write(report)
    f.write("\n=== Średnie metryki z CV ===\n")
    f.write(f"Accuracy: {avg_acc:.4f}\n")
    f.write(f"Precision (macro): {avg_prec:.4f}\n")
    f.write(f"Recall (macro): {avg_rec:.4f}\n")
    f.write(f"F1 Score (macro): {avg_f1:.4f}\n")
    f.write(f"\nCzas treningu: {training_time:.2f} sekund\n")

print("\n📊 Wyniki końcowe (uśrednione):")
print('|====================|')
print(f"Accuracy: {avg_acc:.4f}")
print(f"Precision: {avg_prec:.4f}")
print(f"Recall: {avg_rec:.4f}")
print(f"F1: {avg_f1:.4f}")
print(f"Czas treningu: {training_time:.2f} s")
print('|====================|')



🔁 Fold 1
✅ Fold 1 — Acc: 0.9898, Prec: 0.9738, Rec: 0.9718, F1: 0.9726

🔁 Fold 2
✅ Fold 2 — Acc: 0.9915, Prec: 0.9840, Rec: 0.9805, F1: 0.9813

🔁 Fold 3
✅ Fold 3 — Acc: 0.9872, Prec: 0.9688, Rec: 0.9704, F1: 0.9673

🔁 Fold 4
✅ Fold 4 — Acc: 0.9915, Prec: 0.9727, Rec: 0.9666, F1: 0.9690

🔁 Fold 5
✅ Fold 5 — Acc: 0.9902, Prec: 0.9817, Rec: 0.9704, F1: 0.9746

📊 Wyniki końcowe (uśrednione):
|====================|
Accuracy: 0.9900
Precision: 0.9762
Recall: 0.9719
F1: 0.9729
Czas treningu: 187.82 s
|====================|


In [22]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
import os
from joblib import load
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder

# === ŚCIEŻKI ===
MODEL_PATH = 'models/torch_ssn_model.pt'
ENCODER_PATH = 'models/label_encoder_classes_pt.npy'
SCALER_PATH = 'models/scaler_pt.pkl'
TEST_DATA_PATH = 'test_data.xlsx'

# === 1. Wczytaj model, skaler i encoder ===
if not all(os.path.exists(p) for p in [MODEL_PATH, ENCODER_PATH, SCALER_PATH]):
    raise FileNotFoundError("Brakuje modelu, encodera lub skalera.")

label_classes = np.load(ENCODER_PATH, allow_pickle=True)
label_encoder = LabelEncoder()
label_encoder.classes_ = label_classes

scaler = load(SCALER_PATH)

# === 2. Wczytaj dane testowe ===
df = pd.read_excel(TEST_DATA_PATH)
df = df.drop(columns=[col for col in df.columns if any(x in col for x in ['3_p', '4_p', '5_p'])], errors='ignore')
if 'image_id' in df.columns:
    df = df.drop('image_id', axis=1)

# === 3. Preprocessing ===
def apply_grouped_pca(X, n_components=1):
    lm_0_cols = [col for col in X.columns if col.startswith('0_point_lm_')]
    lm_1_cols = [col for col in X.columns if col.startswith('1_point_lm_')]
    lm_2_cols = [col for col in X.columns if col.startswith('2_point_lm_')]
    vec_cols = [col for col in X.columns if '_vec_' in col]

    def pca_transform(cols, prefix):
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X[cols])
        pca = PCA(n_components=n_components)
        X_pca = pca.fit_transform(X_scaled)
        return pd.DataFrame(X_pca, columns=[f'{prefix}_pca_{i}' for i in range(n_components)], index=X.index)

    pca_lm_0 = pca_transform(lm_0_cols, '0')
    pca_lm_1 = pca_transform(lm_1_cols, '1')
    pca_lm_2 = pca_transform(lm_2_cols, '2')
    vec_features = X[vec_cols].reset_index(drop=True)

    return pd.concat([pca_lm_0, pca_lm_1, pca_lm_2, vec_features], axis=1)
X_test = apply_grouped_pca(df, n_components=1)
X_test_scaled = scaler.transform(X_test)
X_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

# === 4. Wczytaj model i wykonaj predykcję ===
# Najpierw musisz odbudować architekturę modelu
class MLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(MLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.Tanh(),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.model(x)

model = MLP(input_dim=X_test.shape[1], num_classes=len(label_classes))
model.load_state_dict(torch.load(MODEL_PATH, map_location=torch.device('cpu')))
model.eval()

# === 5. Predykcja ===
with torch.no_grad():
    outputs = model(X_tensor)
    probabilities = F.softmax(outputs, dim=1).numpy()

# === 6. Przykład: Prawdopodobieństwa dla pierwszej próbki ===
first_sample_proba = probabilities[0]
class_labels = label_encoder.inverse_transform(np.arange(len(first_sample_proba)))
results = dict(zip(class_labels, np.round(first_sample_proba, 4)))

# === 7. Wynik ===
print('\n📊 Prawdopodobieństwa klas dla pierwszej próbki:')
for label, prob in results.items():
    print(f"{label}: {prob:.4f}")



📊 Prawdopodobieństwa klas dla pierwszej próbki:
a: 0.0021
a+: 0.9974
b: 0.0000
c: 0.0000
c+: 0.0000
ch: 0.0000
cz: 0.0000
d: 0.0000
e: 0.0000
e+: 0.0000
f: 0.0000
g: 0.0000
h: 0.0000
i: 0.0000
j: 0.0000
k: 0.0000
l: 0.0000
l+: 0.0000
m: 0.0000
n: 0.0000
n+: 0.0000
o: 0.0000
o+: 0.0000
p: 0.0000
r: 0.0000
rz: 0.0000
s: 0.0000
s+: 0.0000
sz: 0.0000
t: 0.0000
u: 0.0000
w: 0.0000
y: 0.0000
z: 0.0004
z+: 0.0000
z-: 0.0000


C:\Users\PC2\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\decomposition\_pca.py:586: RuntimeWarning: invalid value encountered in divide
  explained_variance_ = (S**2) / (n_samples - 1)
C:\Users\PC2\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\decomposition\_pca.py:586: RuntimeWarning: invalid value encountered in divide
  explained_variance_ = (S**2) / (n_samples - 1)
C:\Users\PC2\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\decomposition\_pca.py:586: RuntimeWarning: invalid value encountered in divide
  explained_variance_ = (S**2) / (n_samples - 1)
